# CNN por segundo para el FST

**Versión del código: fst_cnn_seg 0.2.0 (red fst_cnn 5.0.0, entrada fusion)**

Corre las celdas **en orden**, una por una (Shift+Enter).

Esta versión deja **figuras listas para la documentación**: datos y desbalance,
muestras de fotogramas por conducta, curva de pérdida en vivo mientras entrena,
matriz de confusión, precisión / recall / F1 por conducta, líneas de tiempo y
ejemplos de errores. Todas se guardan en `Mi unidad/fst/figuras/` (PNG a 200 dpi).

Antes de empezar:
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.
2. En la PC: `py cnn/recortar_seg.py`. Deja la carpeta `clips_seg`.
3. En tu Google Drive, dentro de `Mi unidad/fst/`, sube:
   - la carpeta `clips_seg`
   - `cnn/fst_cnn.py` **y** `cnn/fst_cnn_seg.py` (el segundo usa el primero)

## 1. Comprobar que hay GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY. Cambia el entorno a GPU (ver arriba)')

## 2. Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/fst'
CLIPS = CARPETA + '/clips_seg'
FIGURAS = CARPETA + '/figuras'      # aquí se guardan los PNG y la tabla de métricas
import os, glob
print(len(glob.glob(CLIPS + '/*.npz')), 'videos encontrados en', CLIPS)

## 3. Cargar el código

In [ ]:
import sys, importlib
sys.path.insert(0, CARPETA)
import fst_cnn, fst_cnn_seg
importlib.reload(fst_cnn)
importlib.reload(fst_cnn_seg)

VERSION_ESPERADA = '0.2.0'   # la de este cuaderno
print('Este cuaderno espera fst_cnn_seg', VERSION_ESPERADA)
if fst_cnn_seg.VERSION_SEG != VERSION_ESPERADA:
    print('OJO: Drive tiene la version', fst_cnn_seg.VERSION_SEG, '- sube de nuevo cnn/fst_cnn.py y cnn/fst_cnn_seg.py')
else:
    print('OK: Drive tiene la misma version')

## 4. Los datos, antes de entrenar

Tabla y figura del **desbalance entre conductas**, y fotogramas reales de lo que
ve la red. Esto no entrena nada: tarda un minuto y sirve para la sección de
datos de la documentación.

In [ ]:
X, d, _ = fst_cnn_seg.cargar(CLIPS)
tabla_datos = fst_cnn_seg.figura_datos(d, FIGURAS)
tabla_datos

In [ ]:
# 2 segundos al azar de cada conducta (de videos distintos si se puede), con 4
# fotogramas repartidos en la ventana de 4 s. Cambia semilla para ver otros.
fst_cnn_seg.figura_muestras(X, d, ejemplos=2, fotogramas=4, semilla=0, carpeta_fig=FIGURAS)

## 5. Validar por video (con la curva de pérdida en vivo)

Cada video se predice con una red que **nunca lo vio**. Solo cuenta lo que tú
marcaste en revisor_segundos, así que esta nota sí es honesta.

La curva de pérdida se redibuja después de cada época, una línea por parte de la
validación. Si baja y se aplana, la red aprendió; si una parte queda muy arriba de
las demás, ese video es difícil.

Con ~15 espécimenes tarda alrededor de una hora. Si Colab se corta, cambia a
`cada=2`: entrena con la mitad de los segundos y tarda la mitad.

In [ ]:
fst_cnn_seg.activar_curva_viva()
P, d = fst_cnn_seg.validar(CLIPS, partes=5, cada=1)

## 6. Figuras de resultados para la documentación

Precisión, recall y F1 **por conducta** (con la meta de 0.85 marcada), matriz de
confusión, segundos por espécimen del humano contra la red, qué pasa al descartar
los segundos dudosos, líneas de tiempo humano contra red y ejemplos de errores.

In [ ]:
fst_cnn_seg.figura_entrenamiento(carpeta_fig=FIGURAS)      # curva de pérdida final, por parte y media
tabla_metricas = fst_cnn_seg.figuras_resultados(P, d, X, carpeta_fig=FIGURAS)
tabla_metricas

La tabla también queda en `figuras/metricas_por_conducta.csv`. Las notas de arriba
son **segundo contra segundo, solo sobre lo que marcaste tú**. Mira el recuadro de
"Qué pasa al descartar los segundos dudosos" antes de citar una cifra.

## 7. Entrenar la red final con todo y guardarla en Drive

In [ ]:
fst_cnn_seg.activar_curva_viva()
fst_cnn_seg.final(CLIPS, CARPETA + '/fst_cnn_seg.pt', cada=1)
fst_cnn_seg.figura_entrenamiento({'final': fst_cnn_seg.HISTORIA['final']}, FIGURAS)

Después descarga `fst_cnn_seg.pt` y la carpeta `figuras` de Drive y, en la PC:

    py cnn/probar_seg.py etiquetas/NUEVO_rasgos.csv --red fst_cnn_seg.pt